In [ ]:
# Install required machine learning, ONNX export, and Excel reading tools
!pip install -q imbalanced-learn xgboost shap skl2onnx onnxruntime openpyxl

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
import pandas as pd

# 1. Search for the Excel file in Colab
target_file = None
for root, dirs, files in os.walk("/content"):
    for file in files:
        if "without_infertility" in file.lower() and file.endswith(".xlsx"):
            target_file = os.path.join(root, file)
            break

if not target_file:
    print("❌ File 'PCOS_data_without_infertility.xlsx' not found. Please upload it to the Colab files panel on the left.")
else:
    # 2. Read 'Full_new' (Sheet 2) instead of the Instructions sheet
    try:
        df = pd.read_excel(target_file, sheet_name="Full_new")
    except:
        df = pd.read_excel(target_file, sheet_name=1) # Fallback to second sheet by index

    # Clean whitespace from column titles
    df.columns = df.columns.str.strip()
    print(f"✓ Successfully loaded clinical sheet! Total rows: {len(df)} | Total columns: {len(df.columns)}")

In [ ]:
# Define target features for mobile questionnaire
requested_features = [
    'Age (yrs)', 'BMI', 'Cycle length(days)', 'Cycle(R/I)',
    'Weight gain(Y/N)', 'hair growth(Y/N)', 'Skin darkening (Y/N)',
    'Pimples(Y/N)', 'Fast food (Y/N)', 'Reg.Exercise(Y/N)'
]

app_features = []
for req in requested_features:
    keyword = req.split('(')[0].strip().lower()
    for col in df.columns:
        if keyword in col.lower() and 'pcos' not in col.lower():
            app_features.append(col)
            break

target_col = [c for c in df.columns if 'pcos' in c.lower()][0]

X = df[app_features].apply(pd.to_numeric, errors='coerce').fillna(df.median(numeric_only=True))
y = pd.to_numeric(df[target_col], errors='coerce').fillna(0).astype(int)

print(f"✓ Target Column: '{target_col}'")
print(f"✓ Extracted {len(app_features)} questionnaire features:")
for feat in app_features:
    print(f"  • {feat}")

In [ ]:
from imblearn.over_sampling import SMOTE
from sklearn.model_selection import train_test_split

# Balance target class distribution on raw inputs
smote = SMOTE(random_state=42)
X_resampled, y_resampled = smote.fit_resample(X, y)

# Split 80% train / 20% test
X_train, X_test, y_train, y_test = train_test_split(
    X_resampled, y_resampled, test_size=0.2, random_state=42, stratify=y_resampled
)

print("✓ Data balanced with SMOTE.")
print(f"  • Training samples: {len(X_train)} | Testing samples: {len(X_test)}")

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, HistGradientBoostingClassifier, StackingClassifier
from sklearn.linear_model import LogisticRegression

# Define base ensemble models
base_estimators = [
    ('rf', RandomForestClassifier(n_estimators=150, max_depth=6, random_state=42)),
    ('gb', GradientBoostingClassifier(n_estimators=100, learning_rate=0.05, max_depth=4, random_state=42)),
    ('hgb', HistGradientBoostingClassifier(max_iter=100, learning_rate=0.05, max_depth=4, random_state=42))
]

stacking_model = StackingClassifier(
    estimators=base_estimators,
    final_estimator=LogisticRegression(),
    cv=5
)

# Embed StandardScaler directly into Pipeline
full_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('classifier', stacking_model)
])

# Train full pipeline on unscaled X_train
full_pipeline.fit(X_train, y_train)
print("✓ Unified Pipeline (Scaler + Stacking Classifier) trained successfully!")

In [ ]:
from sklearn.metrics import accuracy_score, classification_report, roc_auc_score

y_pred = full_pipeline.predict(X_test)
y_prob = full_pipeline.predict_proba(X_test)[:, 1]

print("=== DIAGNOSTIC MODEL RESULTS ===")
print("Accuracy Score :", round(accuracy_score(y_test, y_pred) * 100, 2), "%")
print("ROC-AUC Score  :", round(roc_auc_score(y_test, y_prob), 4))
print("\n=== CLASSIFICATION REPORT ===")
print(classification_report(y_test, y_pred))

In [ ]:
import shap

# Scale test set using pipeline's built-in scaler
scaler_step = full_pipeline.named_steps['scaler']
X_test_scaled = scaler_step.transform(X_test)

gb_model = full_pipeline.named_steps['classifier'].named_estimators_['gb']
explainer = shap.TreeExplainer(gb_model)
shap_values = explainer.shap_values(X_test_scaled)

shap.summary_plot(shap_values, X_test, feature_names=app_features)

In [ ]:
import skl2onnx
from skl2onnx.common.data_types import FloatTensorType
from google.colab import files

# Define ONNX input format
initial_type = [('float_input', FloatTensorType([None, len(app_features)]))]

# Disable ZipMap to output raw probability matrix [P(No PCOS), P(PCOS)]
options = {id(full_pipeline): {'zipmap': False}}

onnx_model = skl2onnx.convert_sklearn(
    full_pipeline,
    initial_types=initial_type,
    options=options
)

# Save binary to disk and download
with open("pcos_app_model.onnx", "wb") as f:
    f.write(onnx_model.SerializeToString())

files.download("pcos_app_model.onnx")
print("✓ Production ONNX model downloaded successfully!")